# Werewolf Agent Batch Analysis

Load a batch JSONL file and explore computed metrics across configs.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## 1. Load batch results

In [ ]:
BATCH_FILE = REPO_ROOT / "batch_results" / "v4_action_phase.jsonl"  # <-- change this

records = []
with BATCH_FILE.open() as f:
    for line in f:
        r = json.loads(line.strip())
        if r.get("status") == "success" and "computed_metrics" in r:
            row = {
                "config_name": r["config_name"],
                "run_id": r.get("run_id"),
                "duration_seconds": r.get("duration_seconds"),
                **r["computed_metrics"],
            }
            records.append(row)

df = pd.DataFrame(records)
print(f"Loaded {len(df)} games from {BATCH_FILE.name}")
df.head()

## 2. Win rates by config

In [ ]:
win_rates = df.groupby("config_name")["winner"].value_counts(normalize=True).unstack(fill_value=0)
win_rates.plot.bar(title="Win Rate by Config", ylabel="Win Rate")

## 3. Numeric metrics by config

In [ ]:
numeric_cols = [
    "game_length", "mislynches", "healer_save_count",
    "investigator_wolves_found", "power_roles_killed_by_wolves",
    "tie_count", "no_vote_count",
]
df.groupby("config_name")[numeric_cols].agg(["mean", "std"]).round(2)

## 4. Rate metrics by config

In [ ]:
rate_cols = [
    "correct_elimination_rate", "healer_save_rate", "investigator_accuracy",
    "wolf_steering_rate", "wolf_blending_rate", "wolf_dissent_rate",
    "wolf_power_role_targeting_rate",
]
df.groupby("config_name")[rate_cols].agg(["mean", "count"]).round(3)

## 5. Power role exit methods

In [ ]:
for col in ["healer_exit_method", "investigator_exit_method"]:
    print(f"\n{col}:")
    print(df.groupby("config_name")[col].value_counts().unstack(fill_value=0))

## 6. Metric distributions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.flat, ["game_length", "mislynches", "wolf_steering_rate", "investigator_accuracy"]):
    for config, group in df.groupby("config_name"):
        values = group[col].dropna()
        if not values.empty:
            ax.hist(values, alpha=0.6, label=config, bins=10)
    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.show()

## 7. Full summary table (LLM-friendly)

Copy this output and paste it into an LLM for further analysis.

In [ ]:
from scripts.analyze_batch import analyze, render_markdown, load_records

records_raw = load_records([BATCH_FILE])
analysis = analyze(records_raw)
print(render_markdown(analysis, BATCH_FILE.name))